In [ ]:
%pip install datasets
%pip install torchcodec
%pip install tqdm
%pip install silero-vad
%pip install huggingface-hub

In [10]:

import subprocess
import re
import io
from typing import Tuple, Optional
from datasets import load_dataset, Dataset, Audio, Features, DatasetDict, Value
from tqdm import tqdm
import gc
import json
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from multiprocessing import cpu_count
import threading
from huggingface_hub import notebook_login
import numpy as np
import torch
import os

In [11]:
notebook_login()

In [ ]:
model, utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False,
)
(get_speech_timestamps, _, read_audio, *_) = utils

VAD_SAMPLE_RATE = 16000 



Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


In [ ]:
waxal_tts = [
    'ach',
    'ful',
    'fat',
    'hau', 'ibo',
    'kik',
    'lug', 'luo', 'nyn', 'swa', 'twi', 'yor'
]
##ful, kik
## unique id for each language
waxal_speaker_codes = [
    '100',
    '200',
    '300',
    '400', '500', 
    '600', 
    '700', '800', '900', '1000', '1100', '1200' 
]

source_ds    = 'google/WaxalNLP'
dataset_name = 'evie-8/waxal'
splits = [
    'validation',
    'test',
    'train',
]



In [21]:
def trim_nonspeech(
    audio: np.ndarray,
    sample_rate: int = VAD_SAMPLE_RATE,
    threshold: float = 0.5,
) -> np.ndarray:
    """
    Trim non-speech from the beginning and end of an audio sample using
    Silero VAD.

    Args:
        audio:       1-D numpy array of audio samples (float32, –1..1).
        sample_rate: Sample rate – must be 16 000 Hz for Silero VAD.
        threshold:   VAD confidence threshold (0–1, default 0.5).

    Returns:
        Trimmed 1-D numpy array.  Returns the original array unchanged if
        no speech is detected.
    """
    audio_tensor = torch.from_numpy(audio).float()
    speech_timestamps = get_speech_timestamps(
        audio_tensor,
        model,
        sampling_rate=sample_rate,
        threshold=threshold,
    )

    if not speech_timestamps:
        return audio  # nothing detected – keep original

    start = speech_timestamps[0]['start']
    end   = speech_timestamps[-1]['end']
    return audio[start:end]


In [22]:

def decode_audio_bytes_to_numpy(audio_bytes: bytes, target_sr: int = VAD_SAMPLE_RATE) -> np.ndarray:
    """
    Decode raw audio bytes (any format FFmpeg understands) into a float32
    numpy array resampled to *target_sr* Hz, suitable for Silero VAD.
    """
    cmd = [
        "ffmpeg", "-hide_banner", "-loglevel", "error",
        "-i", "pipe:0",
        "-ar", str(target_sr),
        "-ac", "1",              # mono
        "-f", "f32le",           # raw 32-bit float PCM, little-endian
        "pipe:1",
    ]
    result = subprocess.run(
        cmd, input=audio_bytes,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False,
    )
    if result.returncode != 0:
        raise ValueError(f"FFmpeg decode failed: {result.stderr.decode()[:200]}")
    return np.frombuffer(result.stdout, dtype=np.float32)


def numpy_to_ogg_bytes(audio: np.ndarray, sample_rate: int = VAD_SAMPLE_RATE) -> Tuple[bytes, float]:
    """
    Encode a float32 numpy array as OGG/Opus (falling back to OGG/Vorbis).
    Returns (ogg_bytes, duration_seconds).
    """
    raw_pcm = audio.astype(np.float32).tobytes()

    def _run(codec):
        return subprocess.run(
            [
                "ffmpeg", "-hide_banner", "-loglevel", "info",
                "-f", "f32le", "-ar", str(sample_rate), "-ac", "1",
                "-i", "pipe:0",
                "-c:a", codec, "-b:a", "64k",
                "-f", "ogg", "pipe:1",
            ],
            input=raw_pcm,
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False,
        )

    result = _run("libopus")
    if result.returncode != 0:
        stderr = result.stderr.decode()
        if "libopus" in stderr or "Encoder not found" in stderr:
            result = _run("libvorbis")

    if result.returncode != 0:
        raise ValueError(f"FFmpeg encode failed: {result.stderr.decode()[:200]}")

    ogg_bytes = result.stdout
    if len(ogg_bytes) < 100:
        raise ValueError(f"FFmpeg produced suspiciously small output ({len(ogg_bytes)} bytes)")

    # Parse duration from stderr
    stderr  = result.stderr.decode()
    duration = None
    for pattern in (r'Duration: (\d+):(\d+):(\d+\.\d+)', r'Duration: (\d+):(\d+):(\d+)'):
        m = re.search(pattern, stderr)
        if m:
            h, mi, s = int(m.group(1)), int(m.group(2)), float(m.group(3))
            duration = h * 3600 + mi * 60 + s
            break

    if not duration or duration <= 0:
        duration = (len(ogg_bytes) * 8) / (64000 * 1.1)

    if duration <= 0 or duration > 36000:
        raise ValueError(f"Invalid duration: {duration}s")

    return ogg_bytes, duration


def ffmpeg_bytes_to_ogg(audio_bytes: bytes) -> Tuple[bytes, float]:
    """
    Legacy helper: convert raw audio bytes → OGG **without** VAD trimming.
    Kept for compatibility; prefer the VAD pipeline below.
    """
    audio_np = decode_audio_bytes_to_numpy(audio_bytes)
    return numpy_to_ogg_bytes(audio_np)



In [ ]:

def process_example_batched(batch, speaker_code):
    """
    Process a batch of examples:
      1. Decode audio bytes to 16 kHz float32 numpy array
      2. Run Silero VAD to strip leading/trailing non-speech
      3. Re-encode trimmed audio as OGG
      4. Store result back in the 'audio' column as {'bytes': ..., 'path': None}
    """
    result = {key: [] for key in batch.keys()}

    for i in range(len(batch['audio'])):
        try:
            audio_data = batch['audio'][i]

          
            if audio_data is None:
                for key in result:
                    result[key].append(batch[key][i] if key in batch else None)
                continue

            if isinstance(audio_data, dict):
                if 'bytes' not in audio_data:
                    for key in result:
                        result[key].append(batch[key][i])
                    continue
                audio_bytes = audio_data['bytes']
            elif isinstance(audio_data, bytes):
                audio_bytes = audio_data
            else:
                for key in result:
                    result[key].append(batch[key][i])
                continue

            if not audio_bytes or len(audio_bytes) == 0:
                for key in result:
                    result[key].append(batch[key][i])
                continue

            audio_np  = decode_audio_bytes_to_numpy(audio_bytes, target_sr=VAD_SAMPLE_RATE)
            trimmed   = trim_nonspeech(audio_np, sample_rate=VAD_SAMPLE_RATE, threshold=0.5)
            ogg_bytes, duration = numpy_to_ogg_bytes(trimmed, sample_rate=VAD_SAMPLE_RATE)

            if ogg_bytes is None or duration <= 0:
                for key in result:
                    result[key].append(batch[key][i])
                continue

            for key in result:
                if key == 'audio':
                    result['audio'].append({'bytes': ogg_bytes, 'path': None})
                elif key == 'speaker_id':
                    raw_id = re.sub(r'[^0-9]', '', str(batch['speaker_id'][i]))  # "Female 1" → "1", "2" → "2" Some had text
                    result['speaker_id'].append(speaker_code + raw_id)
                else:
                    result[key].append(batch[key][i])

        except Exception as e:
            print(f"  ⚠ Sample {i} error: {e}")
            for key in result:
                if len(result[key]) < i + 1:
                    result[key].append(batch[key][i])

    return result


In [ ]:
def process_language_split(lang, speaker_code, split, max_workers=6):
    """Process a single language-split combination."""
    try:
        print(f"Processing: {lang} - {split}")

        dataset = load_dataset(source_ds, f"{lang}_tts", split=split, num_proc=5)
        dataset = dataset.rename_column("locale", "audio_language")

        try:
            dataset = dataset.cast_column("audio", Audio(decode=False))
            print("✓ Disabled audio auto-decoding for efficiency")

            dataset = dataset.map(
                process_example_batched,
                batched=True,
                batch_size=100,
                num_proc=os.cpu_count(),
                fn_kwargs={"speaker_code": speaker_code},
            )
            dataset = dataset.cast_column("speaker_id", Value("int64"))
            dataset.push_to_hub(dataset_name, config_name=lang, split=split)

        except Exception as e:
            print(f"⚠ Map/push failed: {e}")
            import traceback
            traceback.print_exc()

    except Exception as e:
        print(f"✗ Error processing {lang}-{split}: {e}")
        import traceback
        traceback.print_exc()
        return None



In [24]:
def main():
    """Main processing function."""
    for lang, speaker_code in zip(waxal_tts, waxal_speaker_codes):
        for split in splits:
            process_language_split(lang, speaker_code, split)

In [ ]:
main()